# Continuous CLCT comparison: NWCSAF cloud amount vs ICON

The companion notebook (`CLCT_threshold_corrected.ipynb`) goes one way: it binarises ICON's
continuous CLCT at a fitted threshold so it can be scored against the binary NWCSAF mask.
This notebook goes the other way, and uses **no threshold at all**.

`cma` is binary per 3 km pixel, but *averaged over many pixels* it is a continuous cloud
fraction — exactly the quantity CLCT already is. So:

1. **Cloud amount over Switzerland.** Average `cma` over all Swiss cells -> one number per hour,
   continuous, no tuned parameter. Compare with ICON's CLCT averaged over the same cells.
   This is the headline timeseries, and it is directly comparable to the MOVERO octa bias.
2. **3x coarsening.** For a *map* the average has to be local, so both fields are block-averaged
   over 3x3 cells (~9 km). Both fields, identically — coarsening only one would compare a smoothed
   observation against a point forecast and make ICON look spuriously too extreme.

**The one caveat to keep in mind:** a pixel that is 20% cloudy still reads `cma = 1`. So the
satellite cloud amount is structurally *higher* than a true areal fraction, most in broken cloud.
That makes the absolute bias hard to read — but it is a near-constant offset, so it cancels in the
two things this project cares about: the *variation with lead time* and the *801 - 802 difference*.
It is also the same effect that pushed `tau*` to 66/77% rather than 50%.

Everything upstream (conservative regridding, T-15 slot matching, parallax, snow/ice screening)
is unchanged from the threshold notebook and justified there / in `docs/conventions.md`.

In [ ]:
import os
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import shapely
import earthkit.data as ekd

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader

if shapely.__version__ < "2":
    raise RuntimeError(f"needs shapely >= 2, found {shapely.__version__}")

# =============================================================================
# Config
# =============================================================================
EXPS = ["801", "802"]
NWCSAF_DIR = "/scratch/mch/jdelbeke/nwcsaf"
ICON_DIR_T = "/store_new/mch/msopr/jdelbeke/ICON_TST/{exp}/FG25/det"
ICON_GRID_FILE = "/oprusers/osm/opr.inn/data/ICON_INPUT/ICON-CH1-EPS/icon_grid_0001_R19B08_mch.nc"

CACHE_DIR = "/scratch/mch/jdelbeke"
WEIGHTS_CACHE = f"{CACHE_DIR}/regrid_weights_icon_nwcsaf.npz"   # shared with the threshold notebook
SERIES_CACHE = f"{CACHE_DIR}/clct_continuous_series.npz"


def nwcsaf_file(t):
    return f"{NWCSAF_DIR}/MSG_nwcsaf_cosmo1eqc3km_{t:%Y%m%d%H%M}.nc"


def icon_file(exp, t):
    return f"{ICON_DIR_T.format(exp=exp)}/lff{t:%Y%m%d%H}"


START = datetime(2025, 10, 4, 1)
N_HOURS = 144
TIMES = [START + timedelta(hours=h) for h in range(N_HOURS)]

DOMAIN = None            # None = whole NWCSAF grid
MIN_COVERAGE = 0.999     # target cells less covered by ICON than this -> NaN
CHUNK = 200_000

# --- corrections (identical to the threshold notebook) -----------------------
PARALLAX = True
NWCSAF_SLOT_LAG = timedelta(minutes=15)   # verified; see docs/conventions.md
CT_SNOW_ICE = (3, 4)                      # Snow_over_land, Sea_ice -- the only screen
SAT_LON = 0.0

# --- coarsening --------------------------------------------------------------
COARSEN = 5              # 5x5 block average, ~15 km, applied to BOTH fields
# 3x was tried first and rejected: a 3x3 block of binary pixels takes only 10
# distinct values, and cloud is spatially correlated, so most blocks came out
# all-cloud or all-clear and the observation still read as a hard mask beside
# ICON's smooth field. 5x5 gives 26 levels. This changes no number in the
# headline timeseries (computed at native 3 km) -- only the maps and the
# coarse RMSE/correlation columns.
MIN_BLOCK_FRAC = 0.5     # a coarse cell needs this fraction of its fine cells valid

OCTA = 12.5              # 1 okta, for the secondary axis against MOVERO

print(f"{len(TIMES)} hourly steps, {TIMES[0]} .. {TIMES[-1]}; experiments {EXPS}")

## 1. Regridding, geometry and the NWCSAF reader

Unchanged from `CLCT_threshold_corrected.ipynb`; the weight cache is shared.

In [ ]:
def read_grid(path, domain=None):
    with xr.open_dataset(path) as ds:
        lon2 = np.asarray(ds["longitude"].values)
        lat2 = np.asarray(ds["latitude"].values)
    if lon2.ndim == 2:
        if (np.abs(lon2 - lon2[0, :]).max() > 1e-9
                or np.abs(lat2 - lat2[:, [0]]).max() > 1e-9):
            raise ValueError("2-D coordinates are not a regular lat/lon grid")
        lon, lat = lon2[0, :], lat2[:, 0]
    else:
        lon, lat = lon2, lat2
    dlon = float(np.median(np.diff(lon)))
    dlat = float(np.median(np.diff(lat)))     # negative: latitude descends
    if (np.abs(np.diff(lon) - dlon).max() > 1e-9
            or np.abs(np.diff(lat) - dlat).max() > 1e-9):
        raise ValueError("grid spacing is not uniform")
    if domain is not None:
        jx = np.where((lon >= domain[0]) & (lon <= domain[1]))[0]
        iy = np.where((lat >= domain[2]) & (lat <= domain[3]))[0]
        lon, lat = lon[jx], lat[iy]
    return lon, lat, dlon, dlat


def read_icon_cells(grid_file, n_expected, lon_c, lat_c):
    with xr.open_dataset(grid_file) as g:
        vlon = np.asarray(g["clon_vertices"].values, dtype="float64")
        vlat = np.asarray(g["clat_vertices"].values, dtype="float64")
    if vlon.shape[0] != n_expected:
        raise ValueError(f"grid file has {vlon.shape[0]} cells, field has {n_expected}")
    if np.abs(vlat).max() <= np.pi + 1e-6:
        vlon, vlat = np.rad2deg(vlon), np.rad2deg(vlat)
    dv = max(np.abs(vlon.mean(axis=1) - lon_c).max(),
             np.abs(vlat.mean(axis=1) - lat_c).max())
    if dv > 1e-3:
        raise ValueError("vertices do not match the field's cell centres")
    return vlon, vlat


def triangle_polygons(vlon, vlat, sub):
    rings = np.stack([vlon[sub], vlat[sub]], axis=-1)
    rings = np.concatenate([rings, rings[:, :1, :]], axis=1)
    return shapely.polygons(rings)


def build_weights(lon, lat, dlon, dlat, vlon, vlat):
    ny, nx = lat.size, lon.size
    ncell = ny * nx
    hx, hy = abs(dlon) / 2, abs(dlat) / 2
    near = ((vlon.max(axis=1) >= lon.min() - hx) & (vlon.min(axis=1) <= lon.max() + hx)
            & (vlat.max(axis=1) >= lat.min() - hy) & (vlat.min(axis=1) <= lat.max() + hy))
    sub = np.where(near)[0]
    tri = triangle_polygons(vlon, vlat, sub)
    LON, LAT = np.meshgrid(lon, lat)
    cells = shapely.box((LON - hx).ravel(), (LAT - hy).ravel(),
                        (LON + hx).ravel(), (LAT + hy).ravel())
    ti, ci = shapely.STRtree(cells).query(tri, predicate="intersects")
    w = np.empty(ti.size)
    for a in range(0, ti.size, CHUNK):
        b = min(a + CHUNK, ti.size)
        w[a:b] = shapely.area(shapely.intersection(tri[ti[a:b]], cells[ci[a:b]]))
    keep = w > 0
    ti, ci, w = ti[keep], ci[keep], w[keep]
    coverage = np.bincount(ci, weights=w, minlength=ncell) / (abs(dlon) * abs(dlat))
    return sub, ti, ci, w, coverage.reshape(ny, nx)


def remap(values, sub, ti, ci, w, coverage, shape):
    v = np.asarray(values, dtype="float64").ravel()[sub]
    ok = np.isfinite(v[ti])
    ncell = shape[0] * shape[1]
    num = np.bincount(ci[ok], weights=w[ok] * v[ti][ok], minlength=ncell)
    den = np.bincount(ci[ok], weights=w[ok], minlength=ncell)
    out = np.full(ncell, np.nan)
    good = (coverage.ravel() >= MIN_COVERAGE) & (den > 0)
    out[good] = num[good] / den[good]
    return out.reshape(shape)

In [ ]:
# --- grid, weights, orography -------------------------------------------------
lon, lat, dlon, dlat = read_grid(nwcsaf_file(TIMES[0]), DOMAIN)
shape = (lat.size, lon.size)

fl = ekd.from_source("file", icon_file(EXPS[0], TIMES[0])).to_fieldlist()
xa = fl.sel({"parameter.variable": "CLCT"})[0].to_xarray()
name0 = "CLCT" if "CLCT" in xa else list(xa.data_vars)[0]
lon_c = np.asarray(xa["longitude"].values).ravel()
lat_c = np.asarray(xa["latitude"].values).ravel()
vlon, vlat = read_icon_cells(ICON_GRID_FILE, lon_c.size, lon_c, lat_c)

if WEIGHTS_CACHE and os.path.exists(WEIGHTS_CACHE):
    z = np.load(WEIGHTS_CACHE)
    sub, ti, ci, w, coverage = z["sub"], z["ti"], z["ci"], z["w"], z["coverage"]
    print(f"weights loaded from {WEIGHTS_CACHE}")
else:
    sub, ti, ci, w, coverage = build_weights(lon, lat, dlon, dlat, vlon, vlat)
    if WEIGHTS_CACHE:
        np.savez_compressed(WEIGHTS_CACHE, sub=sub, ti=ti, ci=ci, w=w, coverage=coverage)
in_domain = coverage >= MIN_COVERAGE

xh = fl.sel({"parameter.variable": "HSURF"})[0].to_xarray()
nmh = "HSURF" if "HSURF" in xh else list(xh.data_vars)[0]
HSURF = remap(np.asarray(xh[nmh].values, dtype="float64").ravel(),
              sub, ti, ci, w, coverage, shape)
HSURF_KM = np.where(np.isfinite(HSURF), HSURF, 0.0) / 1000.0

print(f"target grid {shape}, {int(in_domain.sum())} cells inside the ICON domain")

In [ ]:
# --- parallax geometry and scan timing ---------------------------------------
R_E, R_S = 6371.0, 42164.0
Y_DISC = 0.1518

LON2, LAT2 = np.meshgrid(lon, lat)
_la, _dlo = np.deg2rad(LAT2), np.deg2rad(LON2 - SAT_LON)
PSI = np.arccos(np.cos(_la) * np.cos(_dlo))
VZA = np.arctan2(R_S * np.sin(PSI), R_S * np.cos(PSI) - R_E)
BRG = np.arctan2(np.sin(-_dlo), -np.sin(_la) * np.cos(-_dlo))
DLAT_PER_KM = np.tan(VZA) * np.cos(BRG) / 111.32
DLON_PER_KM = np.tan(VZA) * np.sin(BRG) / (111.32 * np.cos(_la))


def geos_y_angle(lat_deg, lon_deg, sat_lon=SAT_LON):
    c_lat = np.arctan(0.993243 * np.tan(np.deg2rad(lat_deg)))
    r_l = 6356.5838 / np.sqrt(1 - 0.00669438 * np.cos(c_lat) ** 2)
    dl = np.deg2rad(lon_deg - sat_lon)
    r1 = 42164.0 - r_l * np.cos(c_lat) * np.cos(dl)
    r2 = -r_l * np.cos(c_lat) * np.sin(dl)
    r3 = r_l * np.sin(c_lat)
    return np.arcsin(-r3 / np.sqrt(r1 ** 2 + r2 ** 2 + r3 ** 2))


SCAN_FRAC = float((Y_DISC - geos_y_angle(float(np.mean(lat)), float(np.mean(lon))))
                  / (2 * Y_DISC))


def match_slot(t):
    """Strict T-15 pairing; an hour with no such slot is dropped, not re-paired."""
    slot = t - NWCSAF_SLOT_LAG
    return slot if os.path.exists(nwcsaf_file(slot)) else None


_c = (shape[0] // 2, shape[1] // 2)
print(f"viewing zenith at domain centre {np.rad2deg(VZA[_c]):.1f} deg -> "
      f"displacement {np.tan(VZA[_c]):.2f} x height")
print(f"domain imaged {100 * SCAN_FRAC:.0f}% through the sweep; pairing slot "
      f"T-{int(NWCSAF_SLOT_LAG.total_seconds() // 60)}min with model hour T")

In [ ]:
def decode_flags(da):
    a = np.asarray(da.values)
    ai = np.where(np.isfinite(a), a, 0).astype("int64")
    meanings = da.attrs["flag_meanings"].split()
    values = np.asarray(da.attrs["flag_values"]).astype("int64")
    masks = np.asarray(da.attrs.get("flag_mask", da.attrs.get("flag_masks", values))).astype("int64")
    out, seen = {}, {}
    for m, v, mk in zip(meanings, values, masks):
        seen[m] = seen.get(m, 0) + 1
        out[m if seen[m] == 1 else f"{m}__{seen[m]}"] = (ai & int(mk)) == int(v)
    return out


def read_nwcsaf(path, domain=None):
    with xr.open_dataset(path) as ds:
        sl = (slice(None), slice(None))
        cma = np.asarray(ds["cma"].squeeze().values, dtype="float64")[sl]
        cma[cma == 255] = np.nan
        ct = np.asarray(ds["ct"].squeeze().values, dtype="float64")[sl]
        alti = np.asarray(ds["ctth_alti"].squeeze().values, dtype="float64")[sl]
        lo, hi = ds["ctth_alti"].attrs.get("valid_range", (-2000.0, 25000.0))
        alti[(alti < lo) | (alti > hi)] = np.nan
        meth = {k: v[sl] for k, v in decode_flags(ds["ctth_method"]).items()}
        cond = {k: v[sl] for k, v in decode_flags(ds["ctth_conditions"]).items()}
    return dict(cma=cma, ct=ct, alti=alti, meth=meth, cond=cond)


def parallax_correct(sat, hsurf_km):
    """Displace the mask onto true ground positions. Cells that only an
    uncorrectable cloudy pixel could occupy become NaN (unobserved), not clear."""
    cma = sat["cma"]
    ny, nx = cma.shape
    cloudy, clear = cma == 1, cma == 0
    h_km = sat["alti"] / 1000.0
    ok_h = np.isfinite(h_km) & ~sat["meth"].get("No_reliable_method", np.zeros(cma.shape, bool))
    movable, lost = cloudy & ok_h, cloudy & ~ok_h

    def scatter(mask, z_km):
        i0, j0 = np.nonzero(mask)
        lat_t = LAT2[i0, j0] + DLAT_PER_KM[i0, j0] * z_km[i0, j0]
        lon_t = LON2[i0, j0] + DLON_PER_KM[i0, j0] * z_km[i0, j0]
        i1 = np.rint((lat_t - lat[0]) / dlat).astype(int)
        j1 = np.rint((lon_t - lon[0]) / dlon).astype(int)
        keep = (i1 >= 0) & (i1 < ny) & (j1 >= 0) & (j1 < nx)
        cnt = np.zeros((ny, nx), dtype=np.int32)
        np.add.at(cnt, (i1[keep], j1[keep]), 1)
        return cnt

    n_cloud = scatter(movable, np.where(ok_h, h_km, 0.0))
    n_clear = scatter(clear, hsurf_km)
    n_lost = scatter(lost, hsurf_km)
    out = np.full((ny, nx), np.nan)
    out[n_clear > 0] = 0.0
    out[n_cloud > 0] = 1.0                       # cloud wins a collision
    out[(n_lost > 0) & (n_cloud == 0)] = np.nan
    return out


def observed_fraction(slot):
    """NWCSAF cloud amount for one slot, in percent: 0/100 per cell, NaN where
    unobserved. Screened for snow/ice only."""
    S = read_nwcsaf(nwcsaf_file(slot), DOMAIN)
    cma = parallax_correct(S, HSURF_KM) if PARALLAX else S["cma"]
    cma = np.where(np.isin(S["ct"], CT_SNOW_ICE), np.nan, cma)
    return 100.0 * cma, S

## 2. Switzerland mask and the 3x coarsening

The timeseries is a Swiss average, to sit alongside MOVERO's `ch` subset. Both fields are coarsened with the *same* operator and the *same* validity mask, so the comparison stays scale-symmetric.

In [ ]:
_fn = shapereader.natural_earth(resolution="10m", category="cultural",
                                name="admin_0_countries")
_CH_GEOM = [r.geometry for r in shapereader.Reader(_fn).records()
            if r.attributes.get("ADMIN") == "Switzerland"][0]
CH_MASK = shapely.contains_xy(_CH_GEOM, LON2, LAT2) & in_domain


def block_mean(a, k=COARSEN, min_frac=MIN_BLOCK_FRAC):
    """Non-overlapping k x k block average, NaN where too few fine cells are valid."""
    ny, nx = a.shape
    ny2, nx2 = (ny // k) * k, (nx // k) * k
    b = np.asarray(a, dtype="float64")[:ny2, :nx2].reshape(ny2 // k, k, nx2 // k, k)
    v = np.isfinite(b)
    n = v.sum(axis=(1, 3))
    s = np.where(v, b, 0.0).sum(axis=(1, 3))
    out = np.full(n.shape, np.nan)
    ok = n >= max(1, int(np.ceil(min_frac * k * k)))
    out[ok] = s[ok] / n[ok]
    return out


def block_coord(c, k=COARSEN):
    n = (c.size // k) * k
    return c[:n].reshape(-1, k).mean(axis=1)


lon_c3, lat_c3 = block_coord(lon), block_coord(lat)
CH_MASK_C = block_mean(CH_MASK.astype(float)) >= 0.5
shape_c = (lat_c3.size, lon_c3.size)

print(f"Switzerland: {int(CH_MASK.sum())} cells at {abs(dlon) * 111.32 * np.cos(np.deg2rad(46.8)):.1f} km, "
      f"{int(CH_MASK_C.sum())} cells at {COARSEN}x ({COARSEN * abs(dlon) * 111.32 * np.cos(np.deg2rad(46.8)):.1f} km)")
print(f"a coarse cell can take {COARSEN ** 2 + 1} distinct observed values "
      f"(0, 1/{COARSEN ** 2}, ... 1) -- the quantisation floor of this method")

## 3. One pass over the period

Per hour: the Swiss-mean cloud amount from the satellite and from each experiment, over the **same cells** (a cell the satellite could not see is excluded from both). Plus the coarsened pointwise errors.

In [ ]:
# NWCSAF applies a DIFFERENT sequence of spectral tests in each illumination
# regime (day: solar elevation > 10 deg, twilight: -3..10, night: < -3; see the
# CMA product description). So a diurnal curve of NWCSAF-derived bias is a curve
# through three different instruments, and a step at a regime boundary is an
# instrument change, not model behaviour. Every mean below is therefore also
# accumulated separately over the pixels of each regime, using the file's own
# ctth_conditions flags, so that like is only ever compared with like.
REGIMES = ["day", "twilight", "night"]
MIN_REGIME_CELLS = 200      # Swiss cells needed before an hour counts for a regime

# Optimal CLCT thresholds by retrieval condition, fitted on two years of
# OPERATIONAL KENDA-CH1 det FG against NWCSAF (CLCT_tau_conditions.ipynb,
# 16795 hourly scenes, Sep 2024 - Sep 2026). Held-out J showed that adding a
# calendar-month dimension buys only +0.004 J, so these three numbers are the
# whole threshold model. The same taus are applied to 801 and 802: they
# describe the satellite retrieval, not the experiment, so using one set keeps
# the 801-802 difference free of any threshold artefact.
TAU_COND = {"day": 62.0, "twilight": 74.5, "night": 74.0}
TAU_FALLBACK = 68.0         # pixels NWCSAF assigns to none of the three

# Trust flag (section 5c). Pure instrument physics, nothing fitted. The two
# references fail on OPPOSITE cloud types, so the flag says which to prefer:
#   thin ice cloud   -> SEVIRI sees cold ice against a warm background easily,
#                       while the MOVERO reference (APCADA, from longwave
#                       downward radiation) gets almost no signal from optically
#                       thin cirrus. Prefer the satellite.
#   night low cloud  -> a warm-topped deck is a strong APCADA signal but kills
#                       the IR10.8-IR3.9 contrast SEVIRI needs. Prefer the
#                       stations.
# Both branches key off NWCSAF's OWN cloud classes, so no hand-picked height or
# cloud-amount cut is needed. Dropping the cloud-amount condition also removes a
# circularity: selecting hours where the satellite saw little cloud would force
# the satellite-minus-station difference negative by construction.
# The two tests never fire together (0 overlapping hours), so the order is moot.
CIRRUS_FRAC = 0.50    # more than half the CLOUD is thin high cloud
LOW_FRAC = 0.50       # more than half the CLOUD is low cloud, at night
CT_SEMITRANSP = (11, 12, 13, 14, 15)   # genuinely semi-transparent; all 9-10.5 km
CT_LOW = (5, 6)                        # Very_low + Low; tops ~1.4 and ~3.1 km

SIG = "|".join([",".join(EXPS), str(START), str(N_HOURS), str(MIN_COVERAGE),
                str(PARALLAX), str(NWCSAF_SLOT_LAG), str(COARSEN),
                str(MIN_BLOCK_FRAC), str(CT_SNOW_ICE), ",".join(REGIMES),
                str(MIN_REGIME_CELLS), str(sorted(TAU_COND.items())),
                str(TAU_FALLBACK), str((CIRRUS_FRAC, LOW_FRAC)),
                str(CT_SEMITRANSP), str(CT_LOW), "dark=night+twilight"])


def load_clct(exp, t):
    fl_t = ekd.from_source("file", icon_file(exp, t)).to_fieldlist()
    sel = fl_t.sel({"parameter.variable": "CLCT"})
    if len(sel) == 0:
        raise FileNotFoundError("no CLCT field")
    xa_t = sel[0].to_xarray()
    nm = "CLCT" if "CLCT" in xa_t else list(xa_t.data_vars)[0]
    return remap(np.asarray(xa_t[nm].values, dtype="float64").ravel(),
                 sub, ti, ci, w, coverage, shape)


def hour_stats(t):
    """Swiss-mean cloud amount and coarsened errors for one model hour."""
    slot = match_slot(t)
    if slot is None:
        raise FileNotFoundError("no T-15 slot")
    obs, S = observed_fraction(slot)
    clct = {e: load_clct(e, t) for e in EXPS}

    # one common sample: Swiss cells the satellite saw and every experiment covers
    common = CH_MASK & np.isfinite(obs)
    for e in EXPS:
        common &= np.isfinite(clct[e])

    row = {"time": t, "slot": slot, "n_cells": int(common.sum()),
           "unobserved": 1 - common.sum() / max(int(CH_MASK.sum()), 1),
           "obs_mean": float(obs[common].mean())}

    # --- the same means, but over the pixels of one illumination regime only ---
    for rg in REGIMES:
        rm = common & S["cond"].get(rg, np.zeros(shape, bool))
        n_rg = int(rm.sum())
        row[f"n_{rg}"] = n_rg
        ok_rg = n_rg >= MIN_REGIME_CELLS
        row[f"obs_{rg}"] = float(obs[rm].mean()) if ok_rg else np.nan
        for e in EXPS:
            row[f"{e}_{rg}"] = float(clct[e][rm].mean()) if ok_rg else np.nan

    # --- satellite descriptors behind the trust flag ------------------------
    cloudy = common & (obs >= 50.0)
    _a = S["alti"][cloudy & np.isfinite(S["alti"])]
    row["alti_mean"] = float(_a.mean()) if _a.size > 50 else np.nan
    _nc = int(cloudy.sum())
    row["ct_semitransp"] = (float(np.isin(S["ct"][cloudy], CT_SEMITRANSP).mean())
                            if _nc > 50 else np.nan)
    row["ct_low"] = (float(np.isin(S["ct"][cloudy], CT_LOW).mean())
                     if _nc > 50 else np.nan)
    _ncom = max(int(common.sum()), 1)
    row["f_night_px"] = float(
        (common & S["cond"].get("night", np.zeros(shape, bool))).sum() / _ncom)
    # Twilight's missed-cloud rate matches night's, not day's: over two years,
    # M0 (model CLCT where the satellite says CLEAR) is 31.3% at twilight vs
    # 30.6% at night and 24.7% by day, and the fraction of "clear" pixels with
    # model cloud above 80% is 0.184 / 0.182 / 0.120. The marginal solar signal
    # between -3 and +10 deg elevation does not restore detection, so twilight
    # belongs with night here.
    row["f_dark_px"] = row["f_night_px"] + float(
        (common & S["cond"].get("twilight", np.zeros(shape, bool))).sum() / _ncom)

    # --- binary bias: ICON thresholded at its own pixel's condition tau -----
    # Both fields are 0/100 here, so the difference of the means is a
    # cloud-amount bias in the same units as the continuous row above.
    tau_px = np.full(shape, TAU_FALLBACK)
    for rg in REGIMES:
        tau_px[S["cond"].get(rg, np.zeros(shape, bool))] = TAU_COND[rg]
    for e in EXPS:
        row[f"{e}_thr_mean"] = float(
            100.0 * (clct[e][common] >= tau_px[common]).mean())

    # coarsen only the common sample, so both fields see identical blocks
    obs_c = block_mean(np.where(common, obs, np.nan))
    ok_c = CH_MASK_C & np.isfinite(obs_c)
    row["obs_mean_c"] = float(obs_c[ok_c].mean()) if ok_c.any() else np.nan

    for e in EXPS:
        m = clct[e]
        row[f"{e}_mean"] = float(m[common].mean())
        d = m[common] - obs[common]
        row[f"{e}_rmse"] = float(np.sqrt((d ** 2).mean()))
        m_c = block_mean(np.where(common, m, np.nan))
        oc = ok_c & np.isfinite(m_c)
        if oc.sum() > 2:
            dc = m_c[oc] - obs_c[oc]
            row[f"{e}_rmse_c"] = float(np.sqrt((dc ** 2).mean()))
            row[f"{e}_corr_c"] = float(np.corrcoef(m_c[oc], obs_c[oc])[0, 1])
        else:
            row[f"{e}_rmse_c"] = row[f"{e}_corr_c"] = np.nan
    return row


rows = None
if SERIES_CACHE and os.path.exists(SERIES_CACHE):
    _z = np.load(SERIES_CACHE, allow_pickle=True)
    if str(_z["signature"]) == SIG:
        rows = list(_z["rows"])
        print(f"series loaded from {SERIES_CACHE} ({len(rows)} hours)")
    else:
        print("cache built with different settings -- rebuilding")

if rows is None:
    rows = []
    for i, t in enumerate(TIMES, 1):
        try:
            rows.append(hour_stats(t))
        except Exception as e:
            print(f"  skip {t:%m-%d %Hh}: {type(e).__name__}: {e}")
        if i % 24 == 0:
            print(f"  {i}/{len(TIMES)}", flush=True)
    if SERIES_CACHE:
        np.savez_compressed(SERIES_CACHE, rows=np.array(rows, dtype=object),
                            signature=np.array(SIG))

SER = pd.DataFrame(rows).set_index("time").sort_index()
for e in EXPS:
    SER[f"{e}_bias"] = SER[f"{e}_mean"] - SER["obs_mean"]
    SER[f"{e}_bias_thr"] = SER[f"{e}_thr_mean"] - SER["obs_mean"]
    for rg in REGIMES:
        SER[f"{e}_bias_{rg}"] = SER[f"{e}_{rg}"] - SER[f"obs_{rg}"]
SER["hour"] = SER.index.hour


def trust_flag(r):
    """Which observation is expected to fail this hour.

    Two tests, both on NWCSAF's own cloud-type classification, neither using how
    MUCH cloud there was -- so the flag can be validated against the observation
    difference without contaminating it.
    """
    if r["ct_semitransp"] >= CIRRUS_FRAC:
        return "CIRRUS"
    if r["f_dark_px"] > 0.5 and r["ct_low"] >= LOW_FRAC:
        return "STRATUS"
    return "AGREE"


SER["flag"] = SER.apply(trust_flag, axis=1)
print("\ntrust flag: "
      + ",  ".join(f"{k} {v}h" for k, v in SER["flag"].value_counts().items()))

print(f"\n{len(SER)} hours, {SER.index[0]} .. {SER.index[-1]}")
print(f"Swiss cells used per hour: {SER['n_cells'].min()}-{SER['n_cells'].max()} "
      f"of {int(CH_MASK.sum())}  (mean {SER['unobserved'].mean():.1%} unobserved)")
print(f"\nNWCSAF cloud amount over Switzerland: mean {SER['obs_mean'].mean():.1f}%")
for e in EXPS:
    print(f"  {e}: CLCT {SER[f'{e}_mean'].mean():5.1f}%   bias "
          f"{SER[f'{e}_bias'].mean():+5.1f}% ({SER[f'{e}_bias'].mean() / OCTA:+.2f} octa)"
          f"   RMSE {SER[f'{e}_rmse'].mean():.1f}% (3km) -> "
          f"{SER[f'{e}_rmse_c'].mean():.1f}% ({COARSEN}x)"
          f"   corr {SER[f'{e}_corr_c'].mean():.2f}")

print(f"\nthresholded at tau by condition {TAU_COND} "
      f"(fallback {TAU_FALLBACK}%):")
for e in EXPS:
    print(f"  {e}: cloudy fraction {SER[f'{e}_thr_mean'].mean():5.1f}%   bias "
          f"{SER[f'{e}_bias_thr'].mean():+5.1f}% "
          f"({SER[f'{e}_bias_thr'].mean() / OCTA:+.2f} octa)")

## 4. MOVERO first-guess bias, for the panel underneath

In [ ]:
MOVERO_WD = Path("/scratch/mch/jdelbeke/movero/wd/2025_kenda_801_802")
# Repo standard (Okabe-Ito), matching the other notebooks. Do not substitute.
EXP_COLOUR = {"801": "#0072B2", "802": "#D55E00"}
VERIF_SUBSET = "ch"
VERIF_PARAM, VERIF_SCORE = "CLCT", "ME"


def read_movero_time_scores(exp_dir, param=VERIF_PARAM):
    path = Path(exp_dir) / f"time_scores01_{param}.dat"
    with open(path) as fh:
        lines = fh.readlines()
    i_hdr = next(i for i, line in enumerate(lines) if line.split()[:1] == ["YYYY"])
    names = lines[i_hdr].split()
    df = pd.read_csv(path, sep=r"\s+", skiprows=i_hdr + 1, names=names, header=None)
    df = df.mask(df < -1e8)
    df["time"] = pd.to_datetime(dict(year=df["YYYY"], month=df["MM"], day=df["DD"],
                                     hour=df["hh"], minute=df["mm"]))
    return df.set_index("time").sort_index()


bias = {e: read_movero_time_scores(MOVERO_WD / f"{e}-FG-det_{VERIF_SUBSET}")[VERIF_SCORE]
        for e in EXPS}

# MOBS is the station-mean observed cloud cover in octa -- the observation
# itself, identical for both experiments (same stations, same hours), so one
# series is enough. It answers a different question from the bias rows: is the
# model wrong in a way that depends on how cloudy it actually was?
obs_octa = read_movero_time_scores(
    MOVERO_WD / f"{EXPS[0]}-FG-det_{VERIF_SUBSET}")["MOBS"]

# Frozen limits, so the marker sits identically in every frame.
TSPAN = (SER.index[0], SER.index[-1])
_a = np.concatenate([b.dropna().values for b in bias.values()])
MOVERO_YLIM = (np.nanmin(_a) - 0.12 * np.ptp(_a), np.nanmax(_a) + 0.12 * np.ptp(_a))
_b = np.concatenate([SER[f"{e}_bias"].values for e in EXPS])
SAT_YLIM = (np.nanmin(_b) - 0.12 * np.ptp(_b), np.nanmax(_b) + 0.12 * np.ptp(_b))


def draw_sat_bias(ax, t=None, legend=True):
    """ICON CLCT minus NWCSAF cloud amount, Swiss mean, no threshold anywhere."""
    for e in EXPS:
        ax.plot(SER.index, SER[f"{e}_bias"], "-", color=EXP_COLOUR[e], lw=1.3,
                label=f"{e}  CLCT - NWCSAF")
    ax.axhline(0, color="k", lw=0.9)
    ax.set_xlim(*TSPAN)
    ax.set_ylim(*SAT_YLIM)
    ax.grid(alpha=0.3)
    ax.set_ylabel("cloud-amount bias [%]")
    sec = ax.secondary_yaxis("right", functions=(lambda v: v / OCTA, lambda v: v * OCTA))
    sec.set_ylabel("[octa]")
    if legend:
        ax.legend(fontsize=8, ncol=len(EXPS), loc="upper right")
    if t is None:
        return
    ts = pd.Timestamp(t)
    ax.axvline(ts, color="k", lw=1.4, ls=":", zorder=4)
    shown = []
    for e in EXPS:
        v = SER[f"{e}_bias"].get(ts, np.nan)
        if np.isfinite(v):
            ax.plot([ts], [v], marker="*", ms=20, color=EXP_COLOUR[e], mec="k",
                    mew=0.9, zorder=6, clip_on=False)
            shown.append(f"{e} {v:+.1f}%")
    ax.set_title("satellite: Swiss-mean CLCT - NWCSAF cloud amount; "
                 + ("now: " + ",  ".join(shown) if shown else "-"),
                 fontsize=10, loc="left")


def draw_movero_bias(ax, t=None, legend=True):
    for e in EXPS:
        ax.plot(bias[e].index, bias[e].values, "-", color=EXP_COLOUR[e], lw=1.3,
                label=f"{e}  {VERIF_PARAM} {VERIF_SCORE}")
    ax.axhline(0, color="k", lw=0.9)
    ax.set_xlim(*TSPAN)
    ax.set_ylim(*MOVERO_YLIM)
    ax.grid(alpha=0.3)
    ax.set_ylabel(f"{VERIF_PARAM} bias [octa]")
    if legend:
        ax.legend(fontsize=8, ncol=len(EXPS), loc="upper right")
    if t is None:
        return
    ts = pd.Timestamp(t)
    ax.axvline(ts, color="k", lw=1.4, ls=":", zorder=4)
    shown = []
    for e in EXPS:
        v = bias[e].get(ts, np.nan)
        if np.isfinite(v):
            ax.plot([ts], [v], marker="*", ms=20, color=EXP_COLOUR[e], mec="k",
                    mew=0.9, zorder=6, clip_on=False)
            shown.append(f"{e} {v:+.2f}")
    ax.set_title(f"MOVERO first-guess {VERIF_PARAM} bias, all Swiss stations; "
                 + ("now: " + ",  ".join(shown) + " octa" if shown
                    else "no verification at this hour"),
                 fontsize=10, loc="left")


for e in EXPS:
    print(f"{e}: MOVERO {VERIF_SCORE} mean {bias[e].mean():+.3f} octa; "
          f"satellite bias mean {SER[f'{e}_bias'].mean() / OCTA:+.3f} octa")

## 5. The headline figure: two independent views of the same bias

In [ ]:
from matplotlib.patches import Patch

FIGDIR = Path("./figures")
FIGDIR.mkdir(parents=True, exist_ok=True)

_c = np.concatenate([SER[f"{e}_bias_thr"].dropna().values for e in EXPS])
THR_YLIM = (np.nanmin(_c) - 0.12 * np.ptp(_c), np.nanmax(_c) + 0.12 * np.ptp(_c))


def draw_thr_bias(ax, t=None, legend=True):
    """ICON thresholded at the per-condition tau, minus the NWCSAF mask.

    Both fields are binary, so this is the bias the satellite would report if it
    were used as a yes/no ground truth -- the same comparison as the top row, but
    after the model has been put on the satellite's own terms.
    """
    for e in EXPS:
        ax.plot(SER.index, SER[f"{e}_bias_thr"], "-", color=EXP_COLOUR[e], lw=1.3,
                label=f"{e}  CLCT>=tau - NWCSAF")
    ax.axhline(0, color="k", lw=0.9)
    ax.set_xlim(*TSPAN)
    ax.set_ylim(*THR_YLIM)
    ax.grid(alpha=0.3)
    ax.set_ylabel("cloud-amount bias [%]")
    sec = ax.secondary_yaxis("right", functions=(lambda v: v / OCTA, lambda v: v * OCTA))
    sec.set_ylabel("[octa]")
    if legend:
        ax.legend(fontsize=8, ncol=len(EXPS), loc="upper right")
    if t is None:
        return
    ts = pd.Timestamp(t)
    ax.axvline(ts, color="k", lw=1.4, ls=":", zorder=4)
    for e in EXPS:
        v = SER[f"{e}_bias_thr"].get(ts, np.nan)
        if np.isfinite(v):
            ax.plot([ts], [v], marker="*", ms=20, color=EXP_COLOUR[e], mec="k",
                    mew=0.9, zorder=6, clip_on=False)


_tau_txt = ", ".join(f"{k} {v:g}%" for k, v in TAU_COND.items())

# Pale enough to sit under the lines, distinguishable without relying on hue
# alone being read precisely.
FLAG_COLOUR = {"CIRRUS": "#ded4ec", "STRATUS": "#f6dcc0"}
FLAG_LABEL = {"CIRRUS": "over half the cloud is thin high cloud -- APCADA "
                        "under-detects it (prefer the SATELLITE)",
              "STRATUS": "night/twilight, over half the cloud is low -- SEVIRI "
                         "loses IR contrast (prefer the STATIONS)"}


def shade_flags(ax):
    """Paint the hours where one observation is expected to fail."""
    f = SER["flag"]
    step = pd.Timedelta("1h")
    for name, colour in FLAG_COLOUR.items():
        m = (f == name).values
        i = 0
        while i < len(m):
            if not m[i]:
                i += 1
                continue
            j = i
            while j + 1 < len(m) and m[j + 1]:
                j += 1
            ax.axvspan(f.index[i] - 0.5 * step, f.index[j] + 0.5 * step,
                       color=colour, zorder=0, lw=0)
            i = j + 1


def draw_cloudiness(ax, t=None, legend=True):
    """How cloudy it actually was, from both references, in octa."""
    ax.plot(obs_octa.index, obs_octa.values, "-", color="#444444", lw=1.5,
            label="stations (MOVERO MOBS)")
    ax.plot(SER.index, SER["obs_mean"] / OCTA, "-", color="#4a9a6a", lw=1.3,
            label="NWCSAF cloud amount")
    ax.set_xlim(*TSPAN)
    ax.set_ylim(0, 8.4)
    ax.grid(alpha=0.3)
    ax.set_ylabel("observed cloud cover [octa]")
    if legend:
        ax.legend(fontsize=8, ncol=2, loc="lower right")
    if t is not None:
        ax.axvline(pd.Timestamp(t), color="k", lw=1.4, ls=":", zorder=4)


fig, axes = plt.subplots(4, 1, figsize=(16, 14.2), sharex=True)
for _ax in axes:
    shade_flags(_ax)
draw_sat_bias(axes[0])
axes[0].set_title("ICON CLCT minus NWCSAF cloud amount, Swiss mean, 3 km, no threshold",
                  fontsize=11, loc="left")
draw_thr_bias(axes[1])
axes[1].set_title(f"ICON CLCT thresholded by retrieval condition ({_tau_txt}) "
                  f"minus NWCSAF cloud mask, Swiss mean, 3 km", fontsize=11, loc="left")
draw_movero_bias(axes[2])
axes[2].set_title(f"MOVERO first-guess {VERIF_PARAM} bias against station observations, "
                  f"all Swiss stations [octa]", fontsize=11, loc="left")
draw_cloudiness(axes[3])
axes[3].set_title("Observed Swiss-mean cloud cover -- how cloudy it actually was "
                  "(not a bias)", fontsize=11, loc="left")
axes[3].set_xlabel("valid time [UTC]")
fig.suptitle("Swiss cloud-amount bias: satellite continuous, satellite thresholded, "
             "stations -- and the observed cloudiness underneath", fontsize=13)
fig.legend(handles=[Patch(fc=FLAG_COLOUR[k], ec="none", label=FLAG_LABEL[k])
                    for k in ("CIRRUS", "STRATUS")],
           loc="lower center", ncol=2, fontsize=9, frameon=False,
           bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=[0, 0.032, 1, 0.97])
fig.savefig(FIGDIR / "clct_continuous_bias.png", dpi=120)
plt.show()

r, r_thr = {}, {}
for e in EXPS:
    j = pd.concat([SER[f"{e}_bias"] / OCTA, bias[e]], axis=1, join="inner").dropna()
    r[e] = j.corr().iloc[0, 1] if len(j) > 3 else np.nan
    jt = pd.concat([SER[f"{e}_bias_thr"] / OCTA, bias[e]], axis=1, join="inner").dropna()
    r_thr[e] = jt.corr().iloc[0, 1] if len(jt) > 3 else np.nan
print(f"wrote {FIGDIR / 'clct_continuous_bias.png'}")
print("\ncorrelation with the MOVERO station bias "
      "(~4 h decorrelation -> descriptive only):")
for e in EXPS:
    print(f"  {e}: continuous {r[e]:+.2f}   thresholded {r_thr[e]:+.2f}  "
          f"({len(SER)} hours)")

# Does the bias depend on how cloudy it was? Both references, both bias flavours.
print("\ncorrelation of each bias with the observed cloud cover:")
print(f"{'':>6s} {'vs station cloudiness':>24s} {'vs NWCSAF cloudiness':>22s}")
for e in EXPS:
    for lab, y in (("sat thresholded", SER[f"{e}_bias_thr"] / OCTA),
                   ("sat continuous ", SER[f"{e}_bias"] / OCTA),
                   ("MOVERO ME      ", bias[e])):
        j1 = pd.concat([y, obs_octa], axis=1, join="inner").dropna()
        j2 = pd.concat([y, SER["obs_mean"] / OCTA], axis=1, join="inner").dropna()
        c1 = j1.corr().iloc[0, 1] if len(j1) > 3 else np.nan
        c2 = j2.corr().iloc[0, 1] if len(j2) > 3 else np.nan
        print(f"  {e} {lab} {c1:+18.2f} {c2:+22.2f}")

# Split by how cloudy it was, so a non-linear dependence cannot hide in a
# correlation coefficient.
q = obs_octa.reindex(SER.index)
lo, hi = q.quantile(1 / 3), q.quantile(2 / 3)
print(f"\nmean bias [octa] by observed station cloudiness "
      f"(terciles at {lo:.1f} / {hi:.1f} octa):")
print(f"{'':>6s} {'clear third':>13s} {'middle':>10s} {'cloudy third':>14s}")
for e in EXPS:
    for lab, y in (("sat thresholded", SER[f"{e}_bias_thr"] / OCTA),
                   ("MOVERO ME      ", bias[e].reindex(SER.index))):
        g3 = [y[q <= lo].mean(), y[(q > lo) & (q < hi)].mean(), y[q >= hi].mean()]
        print(f"  {e} {lab} {g3[0]:+10.2f} {g3[1]:+10.2f} {g3[2]:+13.2f}")

## 5b. Illumination regimes: comparing like with like

NWCSAF runs a different test sequence by day, at twilight and at night, so the
satellite-derived bias is only comparable **within** a regime. The point of this section
is that the *level* of the bias jumps between regimes for instrument reasons, while the
**801 - 802 difference** does not -- the detection bias is common-mode and cancels.

In [ ]:
REGIME_COLOUR = {"day": "#f7f2d8", "twilight": "#e6dff0", "night": "#dde4ec"}

n_ok = {rg: int(SER[f"obs_{rg}"].notna().sum()) for rg in REGIMES}
print(f"{'regime':>9s} {'hours':>6s} {'NWCSAF':>8s}"
      + "".join(f" {e + ' bias':>11s}" for e in EXPS) + f" {'801-802':>9s}")
for rg in REGIMES:
    if n_ok[rg] == 0:
        print(f"{rg:>9s} {0:6d}   (no hours with enough cells)")
        continue
    d = SER[f"{EXPS[0]}_bias_{rg}"] - SER[f"{EXPS[1]}_bias_{rg}"]
    print(f"{rg:>9s} {n_ok[rg]:6d} {SER[f'obs_{rg}'].mean():7.1f}%"
          + "".join(f" {SER[f'{e}_bias_{rg}'].mean():+10.1f}%" for e in EXPS)
          + f" {d.mean():+8.1f}%")

# The two numbers that matter: how big the instrument step is, against how stable
# the experiment difference is across that same step.
_lv = [SER[f"{EXPS[0]}_bias_{rg}"].mean() for rg in REGIMES if n_ok[rg]]
_df = [(SER[f"{EXPS[0]}_bias_{rg}"] - SER[f"{EXPS[1]}_bias_{rg}"]).mean()
       for rg in REGIMES if n_ok[rg]]
print(f"\nspread of the {EXPS[0]} bias LEVEL across regimes: "
      f"{max(_lv) - min(_lv):.1f}%  <- instrument, not model")
print(f"spread of the {EXPS[0]}-{EXPS[1]} DIFFERENCE across regimes: "
      f"{max(_df) - min(_df):.1f}%  <- the signal that survives")

# --- diurnal composites -----------------------------------------------------
# Regime of each hour-of-day: whichever regime supplies the most Swiss cells.
hour_regime = {}
for h in range(24):
    sl = SER[SER["hour"] == h]
    if len(sl):
        hour_regime[h] = max(REGIMES, key=lambda rg: sl[f"n_{rg}"].mean())

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

for ax in axes:
    for h, rg in hour_regime.items():
        ax.axvspan(h - 0.5, h + 0.5, color=REGIME_COLOUR[rg], zorder=0, lw=0)

# (a) the bias itself -- readable only within a regime
for e in EXPS:
    y = [np.nanmean([SER.loc[SER["hour"] == h, f"{e}_bias_{hour_regime[h]}"].mean()])
         for h in range(24)]
    axes[0].plot(range(24), y, "o-", color=EXP_COLOUR[e], lw=1.6, ms=4, label=e)
axes[0].axhline(0, color="k", lw=0.9)
axes[0].set_ylabel("cloud-amount bias [%]")
axes[0].set_title("(a) satellite bias by hour, each hour scored only on pixels of its own "
                  "regime\n     steps at a shading boundary are NWCSAF changing algorithm, "
                  "not ICON changing behaviour", fontsize=10, loc="left")
axes[0].legend(fontsize=9, ncol=2, loc="upper right")

# (b) the experiment difference -- common-mode, so it survives the regime change
ds = [np.nanmean(SER.loc[SER["hour"] == h,
                         f"{EXPS[0]}_bias_{hour_regime[h]}"].values
                 - SER.loc[SER["hour"] == h,
                           f"{EXPS[1]}_bias_{hour_regime[h]}"].values)
      for h in range(24)]
axes[1].plot(range(24), np.array(ds) / OCTA, "o-", color="k", lw=1.8, ms=4,
             label=f"satellite  {EXPS[0]}-{EXPS[1]}")
dm = [(bias[EXPS[0]][bias[EXPS[0]].index.hour == h].mean()
       - bias[EXPS[1]][bias[EXPS[1]].index.hour == h].mean()) for h in range(24)]
axes[1].plot(range(24), dm, "s--", color="#888888", lw=1.6, ms=4,
             label=f"MOVERO stations  {EXPS[0]}-{EXPS[1]}")
axes[1].axhline(0, color="k", lw=0.9)
axes[1].set_ylabel(f"{EXPS[0]} - {EXPS[1]} bias [octa]")
axes[1].set_xlabel("hour of day [UTC]")
axes[1].set_title(f"(b) the experiment difference, from two independent observing systems "
                  f"-- common-mode detection bias cancels", fontsize=10, loc="left")
axes[1].legend(fontsize=9, ncol=2, loc="upper right")
axes[1].set_xticks(range(0, 24, 2))
axes[1].set_xlim(-0.5, 23.5)
for ax in axes:
    ax.grid(alpha=0.3)

from matplotlib.patches import Patch
axes[0].legend(handles=[plt.Line2D([], [], color=EXP_COLOUR[e], marker="o", label=e)
                        for e in EXPS]
                       + [Patch(fc=REGIME_COLOUR[rg], label=rg) for rg in REGIMES],
               fontsize=8.5, ncol=5, loc="upper right")
fig.suptitle("Diurnal composite, stratified by NWCSAF illumination regime", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(FIGDIR / "clct_continuous_diurnal_regime.png", dpi=120)
plt.show()
print(f"\nwrote {FIGDIR / 'clct_continuous_diurnal_regime.png'}")

## 6. Frames: coarsened ICON against coarsened NWCSAF

One PNG per hour, steppable with `frame_viewer.py`. Both fields are block-averaged over
`COARSEN x COARSEN` cells, from the same validity mask, so the two maps are at the same
effective resolution. The bias panels underneath carry the marker for the frame's hour.

In [ ]:
import sys, subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed
from matplotlib.figure import Figure
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

FRAME_DIR = Path("./figures/clct_continuous_frames")
FRAME_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DPI, N_WORKERS = 110, 16

# Three distinct claims, and white cannot carry two of them:
#   0% cloud                 - a real observation / forecast of clear sky
#   not observed             - parallax gap or snow/ice screen. Excluded from
#                              every number on the figure, in BOTH fields.
#   outside the ICON domain  - no model data at all.
# "clear" and "not observed" both rendering white is the defect this fixes.
NODATA_COLOUR = "#c98a3a"
AMT_CMAP = plt.get_cmap("Blues").copy()
AMT_CMAP.set_bad(NODATA_COLOUR)
DIF_CMAP = plt.get_cmap("RdBu_r").copy()
DIF_CMAP.set_bad(NODATA_COLOUR)
OUTSIDE_CMAP = ListedColormap(["white"])
DIF_LIM = 60.0

# coarse counterpart of in_domain, so "outside the model domain" can be painted
# back over the amber
IN_DOM_C = block_mean(in_domain.astype(float)) >= 0.5


def mask_outside(ax):
    """Paint cells with no model data white, over the not-observed amber."""
    ax.pcolormesh(lon_c3, lat_c3, np.where(~IN_DOM_C, 1.0, np.nan),
                  transform=ccrs.PlateCarree(), cmap=OUTSIDE_CMAP,
                  shading="nearest", zorder=2)

_ii, _jj = np.where(in_domain)
DOMAIN_EXTENT = [lon[_jj.min()] - 2 * abs(dlon), lon[_jj.max()] + 2 * abs(dlon),
                 lat[_ii.max()] - 2 * abs(dlat), lat[_ii.min()] + 2 * abs(dlat)]


def frame_path(t):
    return FRAME_DIR / f"clct_continuous_{t:%Y%m%d%H}.png"


def draw_frame(t):
    slot = match_slot(t)
    obs, _ = observed_fraction(slot)
    clct = {e: load_clct(e, t) for e in EXPS}
    common = in_domain & np.isfinite(obs)
    for e in EXPS:
        common &= np.isfinite(clct[e])

    obs_c = block_mean(np.where(common, obs, np.nan))
    mod_c = {e: block_mean(np.where(common, clct[e], np.nan)) for e in EXPS}

    nrow, NCOL, FIG_W = len(EXPS), 3, 16.5
    _lon_span = DOMAIN_EXTENT[1] - DOMAIN_EXTENT[0]
    _lat_span = DOMAIN_EXTENT[3] - DOMAIN_EXTENT[2]
    _map_h = (FIG_W / NCOL) * 0.90 * _lat_span / _lon_span
    ROW_H, CB_H, LEG_H, BIAS_H, TOP_H = _map_h + 0.45, 0.45, 0.30, 1.85, 0.40
    fig = Figure(figsize=(FIG_W, nrow * ROW_H + CB_H + LEG_H + 2 * BIAS_H + TOP_H))
    gs = fig.add_gridspec(nrow + 4, NCOL,
                          height_ratios=[ROW_H] * nrow + [CB_H, LEG_H, BIAS_H, BIAS_H])

    out = {"time": t}
    row_axes = {}
    for r_, e in enumerate(EXPS):
        d_c = mod_c[e] - obs_c
        swiss = CH_MASK_C & np.isfinite(d_c)
        out[f"{e}_bias_c"] = float(d_c[swiss].mean()) if swiss.any() else np.nan
        panels = [
            (obs_c, "amt", f"NWCSAF cloud amount, {COARSEN}x ({COARSEN * 3} km)\n"
                           f"slot {slot:%H:%M}   Swiss mean "
                           f"{np.nanmean(np.where(CH_MASK_C, obs_c, np.nan)):.1f}%"),
            (mod_c[e], "amt", f"{e} CLCT, {COARSEN}x ({COARSEN * 3} km)\n"
                              f"Swiss mean "
                              f"{np.nanmean(np.where(CH_MASK_C, mod_c[e], np.nan)):.1f}%"),
            (d_c, "dif", f"{e} - NWCSAF\nSwiss mean {out[f'{e}_bias_c']:+.1f}% "
                         f"({out[f'{e}_bias_c'] / OCTA:+.2f} octa)"),
        ]
        for cix, (f_, kind, title) in enumerate(panels):
            ax = fig.add_subplot(gs[r_, cix], projection=ccrs.PlateCarree())
            if kind == "amt":
                mesh_a = ax.pcolormesh(lon_c3, lat_c3, f_, transform=ccrs.PlateCarree(),
                                       cmap=AMT_CMAP, vmin=0, vmax=100, shading="nearest")
            else:
                mesh_d = ax.pcolormesh(lon_c3, lat_c3, f_, transform=ccrs.PlateCarree(),
                                       cmap=DIF_CMAP, vmin=-DIF_LIM, vmax=DIF_LIM,
                                       shading="nearest")
            mask_outside(ax)
            ax.coastlines("10m", linewidth=0.5, zorder=3)
            ax.add_feature(cfeature.BORDERS, linewidth=0.5, zorder=3)
            ax.set_extent(DOMAIN_EXTENT, crs=ccrs.PlateCarree())
            ax.set_title(title, fontsize=8.5)
            if cix == 0:
                row_axes[r_] = ax

    # thin inset bars, so they do not steal width from the fixed-aspect maps
    for cix, (mesh, lab) in enumerate([(mesh_a, "cloud amount [%]"),
                                       (mesh_a, "cloud amount [%]"),
                                       (mesh_d, f"{EXPS[0]}/{EXPS[1]} - NWCSAF [%]")]):
        host = fig.add_subplot(gs[nrow, cix])
        host.axis("off")
        cax = host.inset_axes([0.15, 0.45, 0.70, 0.22])
        cb = fig.colorbar(mesh, cax=cax, orientation="horizontal")
        cb.set_label(lab, fontsize=8.5, labelpad=2)
        cax.tick_params(labelsize=7.5, pad=1)

    # Own gridspec row. Anchored outside the colour-bar host instead, the
    # legend inflated the layout bounding box and tight_layout paid for it by
    # shrinking all six maps -- and it overlapped the colour-bar label.
    ax_leg = fig.add_subplot(gs[nrow + 1, :])
    ax_leg.axis("off")
    ax_leg.legend(handles=[
        Patch(fc=NODATA_COLOUR, ec="k", lw=0.4,
              label="not observed (parallax gap / snow-ice screen) "
                    "-- excluded from both fields"),
        Patch(fc="white", ec="k", lw=0.4, label="outside the ICON domain")],
        loc="center", ncol=2, fontsize=9, frameon=False)

    ax1 = fig.add_subplot(gs[nrow + 2, :])
    draw_sat_bias(ax1, t)
    ax2 = fig.add_subplot(gs[nrow + 3, :])
    draw_movero_bias(ax2, t)
    ax2.set_xlabel("valid time [UTC]")

    fig.suptitle(f"Continuous CLCT comparison, {t:%Y-%m-%d %H:%M} UTC "
                 f"(maps {COARSEN}x coarsened, both fields; timeseries are Swiss means)",
                 fontsize=13)
    fig.tight_layout(rect=[0.014, 0, 1, 0.975], h_pad=0.4)
    for r_, ax0 in row_axes.items():
        p = ax0.get_position()
        fig.text(0.006, 0.5 * (p.y0 + p.y1), f"exp {EXPS[r_]}", rotation=90,
                 va="center", ha="center", fontsize=12, fontweight="bold")
    fig.savefig(frame_path(t), dpi=FRAME_DPI)
    return out


def _job(t):
    try:
        return t, draw_frame(t), None
    except Exception as e:
        return t, None, f"{type(e).__name__}: {e}"


_times = list(SER.index.to_pydatetime())
print(f"rendering {len(_times)} frames on {N_WORKERS} workers -> {FRAME_DIR.resolve()}")
frame_rows, failed = [], []
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futs = [ex.submit(_job, t) for t in _times]
    for done, fu in enumerate(as_completed(futs), 1):
        t, res, err = fu.result()
        (failed if err else frame_rows).append((t, err) if err else res)
        if err:
            print(f"  FAILED {t:%m-%d %Hh}: {err}", flush=True)
        if done % 24 == 0:
            print(f"  {done}/{len(_times)}", flush=True)

print(f"\n{len(frame_rows)} frames written, {len(failed)} failed")
if frame_rows:
    fr = pd.DataFrame(frame_rows).sort_values("time").set_index("time")
    fr.to_csv(FRAME_DIR / "frame_bias.csv")
    print(f"per-frame coarse bias -> {FRAME_DIR / 'frame_bias.csv'}")

(FRAME_DIR / "provenance.txt").write_text(
    f"experiments {', '.join(EXPS)} (rows)\n"
    f"period      {SER.index[0]} .. {SER.index[-1]} ({len(SER)} hours)\n"
    f"maps        both fields block-averaged {COARSEN}x{COARSEN} (~{COARSEN * 3} km),\n"
    f"            same validity mask, no threshold anywhere\n"
    f"columns     NWCSAF amount | ICON CLCT | difference\n"
    f"observation NWCSAF cma, slot T-{int(NWCSAF_SLOT_LAG.total_seconds() // 60)}min,\n"
    f"            parallax-corrected, snow/ice screened\n"
    f"panel 1     Swiss-mean CLCT - NWCSAF cloud amount [%], 3 km\n"
    f"panel 2     MOVERO {VERIF_PARAM} {VERIF_SCORE} [octa], {VERIF_SUBSET}, FG-det\n"
    f"notebook    CLCT_continuous_comparison.ipynb\n")

fv = Path("frame_viewer.py")
if not fv.exists():
    fv = Path("scripts/frame_viewer.py")
if fv.exists():
    subprocess.run([sys.executable, str(fv), str(FRAME_DIR), "--write-only"], check=False)
    print(f"\nStep through them with:\n  python3 {fv} {FRAME_DIR}")

## 7. Summary

In [ ]:
print("=" * 84)
print(f"Continuous CLCT comparison, {SER.index[0]:%d %b} - {SER.index[-1]:%d %b %Y}, "
      f"{len(SER)} hours, Switzerland")
print("=" * 84)
print(f"NWCSAF cloud amount over Switzerland: {SER['obs_mean'].mean():.1f}% "
      f"(range {SER['obs_mean'].min():.0f}-{SER['obs_mean'].max():.0f}%)")
print(f"mean unobserved after parallax + screening: {SER['unobserved'].mean():.1%}\n")
print(f"{'exp':>5s} {'CLCT':>7s} {'bias':>8s} {'bias':>8s} {'RMSE 3km':>9s} "
      f"{f'RMSE {COARSEN}x':>9s} {'corr':>6s} {'MOVERO':>8s}")
for e in EXPS:
    print(f"{e:>5s} {SER[f'{e}_mean'].mean():6.1f}% {SER[f'{e}_bias'].mean():+7.1f}% "
          f"{SER[f'{e}_bias'].mean() / OCTA:+7.2f}o {SER[f'{e}_rmse'].mean():8.1f}% "
          f"{SER[f'{e}_rmse_c'].mean():8.1f}% {SER[f'{e}_corr_c'].mean():6.2f} "
          f"{bias[e].mean():+7.2f}o")
d = SER[f"{EXPS[0]}_bias"] - SER[f"{EXPS[1]}_bias"]
print(f"\n{EXPS[0]} - {EXPS[1]} cloud-amount bias: {d.mean():+.2f}% "
      f"({d.mean() / OCTA:+.3f} octa), {EXPS[0]} cloudier in {100 * (d > 0).mean():.0f}% of hours")
print(f"MOVERO {EXPS[0]} - {EXPS[1]}: {bias[EXPS[0]].mean() - bias[EXPS[1]].mean():+.3f} octa")
print("\nHours are not independent (~4 h decorrelation): read these as descriptive.")
print("The satellite bias carries a structural positive offset (a 20%-cloudy pixel reads")
print("cma=1), so its LEVEL is not a calibrated bias -- its VARIATION and the 801-802")
print("difference are the usable signals.")

## Notes

- **No threshold is used anywhere in this notebook.** That is the point: `tau*` varied 58-84%
  across the day in the threshold notebook, which is a tuned parameter sitting directly on top of
  the diurnal signal this project is investigating.
- **Coarsening is symmetric.** Both fields are block-averaged with the same operator from the same
  validity mask. Coarsening only the observation would compare a smoothed field against a point
  forecast and bias every variance-sensitive score.
- **No time averaging.** Averaging the satellite over +-15 min was considered and rejected: ICON
  writes hourly and CLCT is instantaneous, so the model cannot be coarsened to match, which breaks
  the symmetry above. The effective smoothing would also be `wind x 30 min` along the flow —
  flow-dependent and anisotropic, so different days would get different smoothing silently.
- **This measures cloud amount, not placement.** A forecast with the right total cloud in the wrong
  place scores perfectly here. That is what MOVERO does too; the threshold notebook answers the
  placement half.
- The satellite series and MOVERO are genuinely independent observations (geostationary imager vs
  surface stations) of the same quantity, so agreement between the two panels is meaningful in a
  way that agreement between two model diagnostics would not be.